In [1]:
import pandas as pd

df_mutants = pd.read_csv('data/mutants.csv')

In [2]:
import json
import pandas as pd
import psycopg2

with open('config/postgres_user.json') as json_file:
    creds = json.load(json_file)

DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "logs"
DB_USER = creds["user"]
DB_PASSWORD = creds["password"]

logs = []

df = None

try:
    with psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    ) as conn:
        with conn.cursor() as cursor:
            cursor.execute(
                "SELECT * FROM submissions WHERE date > '2025-06-01';")
            logs = cursor.fetchall()

            df = pd.DataFrame(logs, columns=[desc[0]
                              for desc in cursor.description])

except Exception as e:
    print("Error:", e)

games_df = None

try:
    with psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    ) as conn:
        with conn.cursor() as cursor:
            cursor.execute("SELECT * FROM battleground_games;")
            logs = cursor.fetchall()

            games_df = pd.DataFrame(
                logs, columns=[desc[0] for desc in cursor.description])

except Exception as e:
    print("Error:", e)

In [3]:
# df['game_id'] = df['game_id'].astype(int)
# df['objective'] = df['objective'].astype(int)

# games_df['game_id'] = games_df['game_id'].astype(int)
# games_df['id'] = games_df['id'].astype(int)

# games_df = games_df.sort_values(by=['class_alias', 'attacker', 'defender'])

# games_df = games_df.loc[~games_df.duplicated(
#     subset=['class_alias', 'attacker', 'defender'], keep='last')]

In [4]:
df['game_id'] = df['game_id'].astype(int)
df['objective'] = df['objective'].astype(int)
df['turn'] = df['turn'].astype(int)

# df = df[df['game_id'] >= 244]

games_df['game_id'] = games_df['game_id'].astype(int)
games_df['id'] = games_df['id'].astype(int)
games_df = games_df[games_df['id'] >= 174]

games_df = games_df[~games_df['attacker'].isin(['gpt-4.1-nano'])]
games_df = games_df[~games_df['defender'].isin(['gpt-4.1-nano'])]

# orderby class_alias, attacker, defender
games_df = games_df.sort_values(by=['class_alias', 'attacker', 'defender'])

# check for duplicates, keep the one with the highest id
games_df = games_df.loc[~games_df.duplicated(
    subset=['class_alias', 'attacker', 'defender'], keep='last')]

In [5]:
import numpy as np

mutants = []

for game_id, game in df.groupby('game_id'):
    if not (game_id in games_df['game_id'].values):
        continue
    game_settings = games_df[games_df['game_id'] == game_id].iloc[0]

    for turn, data in game.groupby('turn'):
        attacker_submissions = data[data['side'] == 'attacker']
        game_alias = game_settings['class_alias']
        attacker = game_settings['attacker']
        defender = game_settings['defender']

        state_counts = attacker_submissions['state'].value_counts()
        forfeit = state_counts.get(2, 0) == 0
        killed_immediately = np.NaN
        killed_later = np.NaN
        surviving = np.NaN

        mutant_id = np.NaN

        if not forfeit:
            submission = attacker_submissions[attacker_submissions['state'] == 2].iloc[0]
            mutant_id = json.loads(submission['codedefenders_response'])[
                'mutant']['mutantId']
            
            diff = json.loads(submission['codedefenders_response'])['mutant']['diff']
            modified_lines = json.loads(submission['codedefenders_response'])['mutant']['modifiedLines']

            mutants.append((game_id, game_alias, attacker, defender,
                            turn, mutant_id, diff, modified_lines))


df_diffs = pd.DataFrame(mutants, columns=[
    'game_id', 'class_alias', 'attacker', 'defender', 'turn', 'mutant_id', 'diff', 'modified_lines'])


In [6]:
df_diffs = df_diffs.merge(df_mutants[['equivalent', 'mutant_id']], on='mutant_id', how='left')

In [7]:
df_diffs.to_json('data/mutants_diffs.json', orient='records', lines=True)

In [10]:
df_diffs[df_diffs['diff'] == ""]['class_alias'].value_counts()

Document                 48
XmlElement               40
Rational                 36
SparseIntArray           28
CaseInsensitiveString    10
ByteVector                6
IntHashMap                1
Name: class_alias, dtype: int64